# Transformer Sentiment Analysis — From Scratch

**Goal:** Build a Transformer encoder for binary sentiment classification on IMDb reviews.

We implement everything ourselves:
- Tokenization & Vocabulary
- Sinusoidal Positional Encoding
- Scaled Dot-Product Self-Attention
- Multi-Head Attention
- Feed-Forward Network
- Transformer Block (with residual connections & LayerNorm)
- Transformer Encoder (stacked blocks)
- Classification Head (mean pooling → Dense → Softmax)

**What we DON'T use:** `nn.Transformer`, `nn.MultiheadAttention`, HuggingFace, or any pretrained weights.

**What we DO use:** `nn.Linear`, `nn.LayerNorm`, `nn.Embedding`, optimizers, loss functions — standard PyTorch primitives.

---

## Sections
1. Setup & Config
2. Dataset Loading
3. Text Preprocessing
4. Vocabulary & Numericalization
5. PyTorch Dataset & DataLoader
6. Embeddings
7. Positional Encoding
8. Self-Attention
9. Multi-Head Attention
10. Feed-Forward Network
11. Transformer Block
12. Transformer Encoder
13. Classification Head & Full Model
14. Training
15. Evaluation
16. Inference

---
## 1. Setup & Config

In [ ]:
# Install / upgrade dependencies (run once)
# The huggingface_hub upgrade fixes the HfUriError with "imdb" short name
!pip install -q --upgrade datasets huggingface_hub
!pip install -q torch scikit-learn matplotlib seaborn tqdm


In [ ]:
import math
import re
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from datasets import load_dataset
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

In [ ]:
# ─── CONFIGURATION ───────────────────────────────────────────────────────────

# Set QUICK_RUN = True for a fast sanity check (1k samples, 2 epochs)
# Set QUICK_RUN = False for the full IMDb run (25k train, 10 epochs)
QUICK_RUN = False

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Data
MAX_LEN       = 200       # max tokens per review
VOCAB_SIZE    = 20_000    # vocabulary size (+ special tokens)
QUICK_SAMPLES = 1_000     # samples per split in QUICK_RUN mode

# Model
EMBED_DIM = 256           # d_model
NUM_HEADS   = 4           # must divide EMBED_DIM evenly
FF_DIM      = 512         # feed-forward hidden size
NUM_BLOCKS  = 3           # number of Transformer blocks
DROPOUT     = 0.3         # ↑ increased from 0.1 — regularizes against overfitting
NUM_CLASSES = 2           # positive / negative

# Training
BATCH_SIZE    = 32
LEARNING_RATE = 1e-4      # ↓ reduced from 1e-4 — smoother, more stable convergence
WEIGHT_DECAY  = 1e-4      # L2 regularization — penalizes large weights
EPOCHS        = 2 if QUICK_RUN else 15  # more epochs since LR is lower
PATIENCE      = 5         # early stopping — halt if val loss stalls for 3 epochs

# Special tokens
PAD_TOKEN = "<PAD>"       # index 0
UNK_TOKEN = "<UNK>"       # index 1

print(f"QUICK_RUN = {QUICK_RUN}")
print(f"Epochs    = {EPOCHS} (+ early stopping with patience={PATIENCE})")
print(f"Dropout   = {DROPOUT}")
print(f"LR        = {LEARNING_RATE}")


---
## 2. Dataset Loading

We use the IMDb dataset from HuggingFace Datasets.  
- 25,000 training reviews  
- 25,000 test reviews  
- Labels: 0 = negative, 1 = positive

In [ ]:
print("Loading IMDb dataset...")

# Newer versions of huggingface_hub require the full namespace/name format.
# "stanfordnlp/imdb" is the canonical namespaced path.
try:
    raw_dataset = load_dataset("stanfordnlp/imdb")
except Exception as e:
    print(f"stanfordnlp/imdb failed ({e}), trying legacy path...")
    raw_dataset = load_dataset("imdb", trust_remote_code=True)

print(raw_dataset)


In [ ]:
# Extract lists
train_texts  = raw_dataset["train"]["text"]
train_labels = raw_dataset["train"]["label"]
test_texts   = raw_dataset["test"]["text"]
test_labels  = raw_dataset["test"]["label"]

# For QUICK_RUN: sample a small balanced subset
def balanced_sample(texts, labels, n_per_class):
    pos_idx = [i for i, l in enumerate(labels) if l == 1][:n_per_class]
    neg_idx = [i for i, l in enumerate(labels) if l == 0][:n_per_class]
    idx = pos_idx + neg_idx
    random.shuffle(idx)
    return [texts[i] for i in idx], [labels[i] for i in idx]

if QUICK_RUN:
    n = QUICK_SAMPLES // 2
    train_texts, train_labels = balanced_sample(train_texts, train_labels, n)
    test_texts,  test_labels  = balanced_sample(test_texts,  test_labels,  n // 2)

# Create a small validation split from training data (15%)
val_size = max(1, int(0.15 * len(train_texts)))
val_texts,   val_labels   = train_texts[:val_size],  train_labels[:val_size]
train_texts, train_labels = train_texts[val_size:],  train_labels[val_size:]

print(f"Train: {len(train_texts)} | Val: {len(val_texts)} | Test: {len(test_texts)}")
print(f"Label distribution (train) — pos: {sum(train_labels)}, neg: {len(train_labels)-sum(train_labels)}")

In [ ]:
# Quick look at a sample
print("Sample review:")
print(train_texts[0][:300])
print(f"\nLabel: {'Positive' if train_labels[0] == 1 else 'Negative'}")

---
## 3. Text Preprocessing

Simple but effective pipeline:
1. Lowercase
2. Remove HTML tags (IMDb reviews often contain `<br />`)
3. Keep only letters, digits, spaces
4. Split on whitespace → token list

In [ ]:
def preprocess(text: str) -> list[str]:
    """Clean and tokenize a review into a list of word tokens."""
    text = text.lower()
    text = re.sub(r"<[^>]+>", " ", text)          # strip HTML tags
    text = re.sub(r"[^a-z0-9\s]", " ", text)       # keep alphanumeric
    text = re.sub(r"\s+", " ", text).strip()        # collapse whitespace
    return text.split()

# Test it
sample = "This movie was <b>absolutely</b> fantastic! I loved it."
print(preprocess(sample))

In [ ]:
print("Tokenizing all splits...")
train_tokens = [preprocess(t) for t in tqdm(train_texts, desc="Train")]
val_tokens   = [preprocess(t) for t in tqdm(val_texts,   desc="Val")]
test_tokens  = [preprocess(t) for t in tqdm(test_texts,  desc="Test")]

# Quick stats
lengths = [len(t) for t in train_tokens]
print(f"\nToken length — mean: {np.mean(lengths):.0f}, median: {np.median(lengths):.0f}, max: {max(lengths)}")

In [ ]:
# Visualize review length distribution
plt.figure(figsize=(8, 3))
plt.hist(lengths, bins=50, color='steelblue', edgecolor='white')
plt.axvline(MAX_LEN, color='red', linestyle='--', label=f'MAX_LEN={MAX_LEN}')
plt.xlabel("Number of tokens")
plt.ylabel("Count")
plt.title("Review Length Distribution (train)")
plt.legend()
plt.tight_layout()
plt.show()

pct_truncated = sum(1 for l in lengths if l > MAX_LEN) / len(lengths) * 100
print(f"{pct_truncated:.1f}% of reviews will be truncated at MAX_LEN={MAX_LEN}")

---
## 4. Vocabulary & Numericalization

Build vocabulary from **training data only** (no data leakage from val/test).

- Index 0 → `<PAD>` (padding)
- Index 1 → `<UNK>` (unknown words)
- Index 2+ → most frequent words

In [ ]:
class Vocabulary:
    def __init__(self, max_size=20_000):
        self.max_size  = max_size
        self.word2idx  = {PAD_TOKEN: 0, UNK_TOKEN: 1}
        self.idx2word  = {0: PAD_TOKEN, 1: UNK_TOKEN}
        self.size      = 2

    def build(self, token_lists):
        """Count all tokens and keep the top (max_size - 2) by frequency."""
        counter = Counter()
        for tokens in token_lists:
            counter.update(tokens)

        most_common = counter.most_common(self.max_size - 2)  # reserve 2 for special tokens
        for word, _ in most_common:
            idx = self.size
            self.word2idx[word] = idx
            self.idx2word[idx]  = word
            self.size += 1

        print(f"Vocabulary built: {self.size} tokens (including <PAD> and <UNK>)")
        return self

    def encode(self, tokens, max_len):
        """Convert token list → padded integer tensor of length max_len."""
        ids = [self.word2idx.get(t, 1) for t in tokens[:max_len]]  # 1 = UNK
        # Pad to max_len
        ids += [0] * (max_len - len(ids))  # 0 = PAD
        return ids


vocab = Vocabulary(max_size=VOCAB_SIZE)
vocab.build(train_tokens)

# Demo
sample_ids = vocab.encode(["this", "movie", "was", "fantastic", "xyz_unknown"], max_len=8)
print("\nSample encoding:", sample_ids)
print("Decoded:        ", [vocab.idx2word.get(i, UNK_TOKEN) for i in sample_ids])

In [ ]:
# Encode all splits
print("Encoding all splits...")
train_ids = [vocab.encode(t, MAX_LEN) for t in tqdm(train_tokens, desc="Train")]
val_ids   = [vocab.encode(t, MAX_LEN) for t in tqdm(val_tokens,   desc="Val")]
test_ids  = [vocab.encode(t, MAX_LEN) for t in tqdm(test_tokens,  desc="Test")]
print("Done.")

---
## 5. PyTorch Dataset & DataLoader

In [ ]:
class IMDbDataset(Dataset):
    def __init__(self, ids, labels):
        self.ids    = torch.tensor(ids,    dtype=torch.long)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.ids[idx], self.labels[idx]


train_dataset = IMDbDataset(train_ids, train_labels)
val_dataset   = IMDbDataset(val_ids,   val_labels)
test_dataset  = IMDbDataset(test_ids,  test_labels)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

# Sanity check
batch_x, batch_y = next(iter(train_loader))
print(f"Batch input shape:  {batch_x.shape}   (batch_size × seq_len)")
print(f"Batch labels shape: {batch_y.shape}")
print(f"Sample label values: {batch_y[:8].tolist()}")

---
## 6. Embeddings

Convert token indices into dense vectors of dimension `EMBED_DIM`.

- Embedding matrix shape: `(vocab_size, embed_dim)`
- We set `padding_idx=0` so PAD tokens always have zero embeddings and don't accumulate gradients.

In [ ]:
class TokenEmbedding(nn.Module):
    """
    Wraps nn.Embedding with a scaling factor (√d_model).
    Scaling is a trick from 'Attention Is All You Need' to keep
    embeddings from being drowned out by positional encodings.
    """
    def __init__(self, vocab_size, embed_dim, padding_idx=0):
        super().__init__()
        self.embed_dim = embed_dim
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=padding_idx)

    def forward(self, x):
        # x: (batch, seq_len) → output: (batch, seq_len, embed_dim)
        return self.embedding(x) * math.sqrt(self.embed_dim)


# Quick test
emb = TokenEmbedding(vocab.size, EMBED_DIM)
dummy_input = torch.randint(0, vocab.size, (2, MAX_LEN))
emb_output  = emb(dummy_input)
print(f"Embedding output shape: {emb_output.shape}")
print(f"  (batch=2, seq_len={MAX_LEN}, embed_dim={EMBED_DIM})")

---
## 7. Positional Encoding

Transformers have no recurrence, so they can't tell the position of a token by themselves.
We add a fixed positional signal to each token embedding.

**Formula** (from *Attention Is All You Need*):

```
PE(pos, 2i)   = sin(pos / 10000^(2i/d_model))
PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))
```

Why alternating sin/cos? Nearby positions have similar encodings and distant positions differ — this gives the model a smooth sense of ordering.

In [ ]:
class PositionalEncoding(nn.Module):
    """
    Sinusoidal positional encoding — fixed, not learned.
    Adds position information to token embeddings.
    """
    def __init__(self, embed_dim, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Build PE matrix: shape (max_len, embed_dim)
        pe  = torch.zeros(max_len, embed_dim)
        pos = torch.arange(0, max_len).unsqueeze(1).float()   # (max_len, 1)

        # Compute the division term: 10000^(2i/d_model)
        div_term = torch.exp(
            torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim)
        )

        pe[:, 0::2] = torch.sin(pos * div_term)   # even indices
        pe[:, 1::2] = torch.cos(pos * div_term)   # odd indices

        # Register as buffer (not a parameter — won't be updated during training)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, embed_dim)

    def forward(self, x):
        # x: (batch, seq_len, embed_dim)
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


# Visualize the positional encoding
pe_module = PositionalEncoding(EMBED_DIM, dropout=0.0)
pe_matrix = pe_module.pe[0].numpy()   # (max_len, embed_dim)

plt.figure(figsize=(10, 4))
plt.imshow(pe_matrix[:100, :64], aspect='auto', cmap='RdBu')
plt.colorbar()
plt.xlabel("Embedding dimension")
plt.ylabel("Position")
plt.title("Sinusoidal Positional Encoding (first 100 positions, 64 dims)")
plt.tight_layout()
plt.show()

---
## 8. Self-Attention (Scaled Dot-Product)

The core of the Transformer. Given a sequence, each token looks at every other token and decides how much attention to pay.

**Steps:**
1. Project input X into Q, K, V using learned weight matrices
2. Compute attention scores: `scores = Q @ K.T / √d_k`
3. Apply softmax to get attention weights
4. Weighted sum of values: `output = weights @ V`

**Intuition:**  
- Q = "what am I looking for?"
- K = "what do I offer?"
- V = "what do I actually return?"

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q: (batch, heads, seq_len, d_k)
    K: (batch, heads, seq_len, d_k)
    V: (batch, heads, seq_len, d_v)
    mask: (batch, 1, 1, seq_len) — True where we want to IGNORE (PAD tokens)

    Returns:
        output:          (batch, heads, seq_len, d_v)
        attention_weights: (batch, heads, seq_len, seq_len)
    """
    d_k = Q.size(-1)

    # Step 1: Compute scores
    scores = torch.matmul(Q, K.transpose(-2, -1))  # (batch, heads, seq_len, seq_len)

    # Step 2: Scale (prevents vanishing gradients for large d_k)
    scores = scores / math.sqrt(d_k)

    # Step 3: Apply mask (set PAD positions to -inf so softmax ignores them)
    if mask is not None:
        scores = scores.masked_fill(mask, float("-inf"))

    # Step 4: Softmax over the key dimension
    attention_weights = F.softmax(scores, dim=-1)  # (batch, heads, seq_len, seq_len)

    # Step 5: Weighted sum of values
    output = torch.matmul(attention_weights, V)    # (batch, heads, seq_len, d_v)

    return output, attention_weights


# Unit test — single head, no mask
B, S, D = 2, 10, 32
Q_test = torch.randn(B, 1, S, D)
K_test = torch.randn(B, 1, S, D)
V_test = torch.randn(B, 1, S, D)
out, attn = scaled_dot_product_attention(Q_test, K_test, V_test)
print(f"Attention output shape:  {out.shape}")
print(f"Attention weights shape: {attn.shape}")
print(f"Weights sum to 1 (row 0, head 0, pos 0): {attn[0,0,0].sum():.4f}")

---
## 9. Multi-Head Attention

Instead of one attention mechanism, we run **H parallel attention heads**, each learning different relationships (e.g., grammar, sentiment, coreference).

```
head_i = Attention(Q·Wq_i, K·Wk_i, V·Wv_i)
MultiHead = Concat(head_1, ..., head_H) · Wo
```

In practice: we project to full `d_model` then *reshape* into H heads — more efficient than H separate linear layers.

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, dropout=0.1):
        super().__init__()
        assert embed_dim % num_heads == 0, \
            f"embed_dim ({embed_dim}) must be divisible by num_heads ({num_heads})"

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.d_k       = embed_dim // num_heads   # dimension per head

        # Single projection for Q, K, V (could also be three separate linears)
        self.W_q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.W_v = nn.Linear(embed_dim, embed_dim, bias=False)

        # Output projection
        self.W_o = nn.Linear(embed_dim, embed_dim)

        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        """
        Reshape (batch, seq_len, embed_dim)
            → (batch, num_heads, seq_len, d_k)
        """
        B, S, _ = x.shape
        x = x.view(B, S, self.num_heads, self.d_k)
        return x.permute(0, 2, 1, 3)  # (B, H, S, d_k)

    def forward(self, x, mask=None):
        """
        x:    (batch, seq_len, embed_dim)
        mask: (batch, 1, 1, seq_len)  — True where PAD
        """
        B, S, _ = x.shape

        # Project
        Q = self.split_heads(self.W_q(x))   # (B, H, S, d_k)
        K = self.split_heads(self.W_k(x))   # (B, H, S, d_k)
        V = self.split_heads(self.W_v(x))   # (B, H, S, d_k)

        # Attend
        attn_out, _ = scaled_dot_product_attention(Q, K, V, mask=mask)
        # attn_out: (B, H, S, d_k)

        # Merge heads: (B, H, S, d_k) → (B, S, embed_dim)
        attn_out = attn_out.permute(0, 2, 1, 3).contiguous()
        attn_out = attn_out.view(B, S, self.embed_dim)

        # Output projection
        output = self.W_o(attn_out)
        output = self.dropout(output)

        return output


# Unit test
mha = MultiHeadAttention(EMBED_DIM, NUM_HEADS)
x_test = torch.randn(4, MAX_LEN, EMBED_DIM)
out_test = mha(x_test)
print(f"MultiHeadAttention output shape: {out_test.shape}")
print(f"  Expected: (4, {MAX_LEN}, {EMBED_DIM})")

---
## 10. Feed-Forward Network (FFN)

A simple 2-layer MLP applied **independently to each position**:

```
FFN(x) = ReLU(x @ W1 + b1) @ W2 + b2
```

The inner dimension is usually 4× the model dimension (here: 128 → 512 → 128).

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, embed_dim, ff_dim, dropout=0.1):
        super().__init__()
        self.linear1 = nn.Linear(embed_dim, ff_dim)
        self.linear2 = nn.Linear(ff_dim, embed_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # x: (batch, seq_len, embed_dim)
        x = F.relu(self.linear1(x))   # → (batch, seq_len, ff_dim)
        x = self.dropout(x)
        x = self.linear2(x)           # → (batch, seq_len, embed_dim)
        return x


# Unit test
ff = FeedForward(EMBED_DIM, FF_DIM)
ff_out = ff(x_test)
print(f"FeedForward output shape: {ff_out.shape}")
print(f"  Expected: (4, {MAX_LEN}, {EMBED_DIM})")

---
## 11. Transformer Block

One complete Transformer encoder block:

```
x → [Multi-Head Attention] → Add & LayerNorm → [FFN] → Add & LayerNorm
```

The **residual connections** (`x + sublayer(x)`) allow gradients to flow back through many layers — without them, deep Transformers would be nearly untrainable.

In [ ]:
class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(embed_dim, num_heads, dropout)
        self.ff        = FeedForward(embed_dim, ff_dim, dropout)

        # LayerNorm normalizes across the embedding dimension (not the batch)
        self.norm1 = nn.LayerNorm(embed_dim)
        self.norm2 = nn.LayerNorm(embed_dim)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        """
        x:    (batch, seq_len, embed_dim)
        mask: (batch, 1, 1, seq_len)
        """
        # Sub-layer 1: Multi-Head Self-Attention + Residual + LayerNorm
        attn_out = self.attention(x, mask=mask)
        x = self.norm1(x + attn_out)    # Add & Norm

        # Sub-layer 2: Feed-Forward + Residual + LayerNorm
        ff_out = self.ff(x)
        x = self.norm2(x + ff_out)      # Add & Norm

        return x


# Unit test
block = TransformerBlock(EMBED_DIM, NUM_HEADS, FF_DIM)
block_out = block(x_test)
print(f"TransformerBlock output shape: {block_out.shape}")
print(f"  Expected: (4, {MAX_LEN}, {EMBED_DIM})")

---
## 12. Transformer Encoder

Stack `NUM_BLOCKS` Transformer blocks sequentially.  
We also create a **padding mask** to prevent the model from attending to `<PAD>` tokens.

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, ff_dim,
                 num_blocks, max_len, dropout=0.1, padding_idx=0):
        super().__init__()
        self.embedding  = TokenEmbedding(vocab_size, embed_dim, padding_idx)
        self.pos_enc    = PositionalEncoding(embed_dim, max_len, dropout)
        self.blocks     = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads, ff_dim, dropout)
            for _ in range(num_blocks)
        ])
        self.padding_idx = padding_idx

    def make_padding_mask(self, x):
        """
        Create a boolean mask where True = PAD (should be ignored).
        Shape: (batch, 1, 1, seq_len) — broadcast over heads and query positions.
        """
        # x: (batch, seq_len)  → mask: (batch, 1, 1, seq_len)
        return (x == self.padding_idx).unsqueeze(1).unsqueeze(2)

    def forward(self, x):
        """
        x: (batch, seq_len)  — integer token ids
        Returns: (batch, seq_len, embed_dim)
        """
        mask = self.make_padding_mask(x)   # (B, 1, 1, S)

        x = self.embedding(x)              # (B, S, E)
        x = self.pos_enc(x)                # (B, S, E)

        for block in self.blocks:
            x = block(x, mask=mask)        # (B, S, E)

        return x                           # (B, S, E)


# Unit test
encoder = TransformerEncoder(
    vocab_size=vocab.size,
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS,
    ff_dim=FF_DIM,
    num_blocks=NUM_BLOCKS,
    max_len=MAX_LEN + 10,
    dropout=DROPOUT
)
ids_test = torch.randint(0, vocab.size, (4, MAX_LEN))
enc_out  = encoder(ids_test)
print(f"TransformerEncoder output shape: {enc_out.shape}")
print(f"  Expected: (4, {MAX_LEN}, {EMBED_DIM})")

---
## 13. Classification Head & Full Model

**Mean pooling:** average all token representations into a single sentence vector.  
We mask PAD tokens out of the mean so they don't contribute.

Then: `sentence_vector → Linear(embed_dim, num_classes) → logits`

In [ ]:
class SentimentClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_heads, ff_dim,
                 num_blocks, max_len, num_classes, dropout=0.1, padding_idx=0):
        super().__init__()
        self.encoder     = TransformerEncoder(
            vocab_size, embed_dim, num_heads, ff_dim,
            num_blocks, max_len, dropout, padding_idx
        )
        self.classifier  = nn.Linear(embed_dim, num_classes)
        self.dropout     = nn.Dropout(dropout)
        self.padding_idx = padding_idx

    def mean_pool(self, token_embeddings, input_ids):
        """
        Average token embeddings, ignoring PAD positions.
        token_embeddings: (batch, seq_len, embed_dim)
        input_ids:        (batch, seq_len)
        Returns:          (batch, embed_dim)
        """
        # Non-PAD mask: (B, S, 1)
        mask = (input_ids != self.padding_idx).unsqueeze(-1).float()
        # Zero out PAD positions
        masked = token_embeddings * mask
        # Sum / count
        summed = masked.sum(dim=1)                  # (B, E)
        counts = mask.sum(dim=1).clamp(min=1e-9)    # (B, 1)
        return summed / counts                      # (B, E)

    def forward(self, x):
        """
        x: (batch, seq_len)  → logits: (batch, num_classes)
        """
        token_out = self.encoder(x)          # (B, S, E)
        pooled    = self.mean_pool(token_out, x)  # (B, E)
        pooled    = self.dropout(pooled)
        logits    = self.classifier(pooled)  # (B, num_classes)
        return logits


# Instantiate the full model
model = SentimentClassifier(
    vocab_size  = vocab.size,
    embed_dim   = EMBED_DIM,
    num_heads   = NUM_HEADS,
    ff_dim      = FF_DIM,
    num_blocks  = NUM_BLOCKS,
    max_len     = MAX_LEN + 10,
    num_classes = NUM_CLASSES,
    dropout     = DROPOUT
).to(DEVICE)

# Count parameters
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters:     {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# Forward pass test
ids_test = torch.randint(0, vocab.size, (4, MAX_LEN)).to(DEVICE)
logits   = model(ids_test)
print(f"\nLogits shape: {logits.shape}   (batch=4, classes=2)")

---
## 14. Training

- **Loss:** `CrossEntropyLoss`
- **Optimizer:** Adam with `lr=3e-5` and `weight_decay=1e-4` (L2 regularization)
- **Scheduler:** `ReduceLROnPlateau` — halves LR if val loss doesn't improve for 2 epochs
- **Early stopping:** halts training if val loss shows no improvement for `PATIENCE=3` epochs
- **Tracking:** train loss, val loss, val accuracy per epoch

### Why these changes vs the original?
| Change | Reason |
|---|---|
| Dropout 0.1 → 0.3 | Randomly drops more neurons, forcing the model to not rely on any single path |
| LR 1e-4 → 3e-5 | Smaller steps = smoother loss landscape, less val loss spiking |
| Weight decay 1e-4 | Penalizes large weights, discourages memorization |
| Early stopping | Automatically saves the best checkpoint and stops before overfitting worsens |


In [ ]:
criterion = nn.CrossEntropyLoss()

# Weight decay adds L2 penalty: effective_loss = loss + weight_decay * sum(w²)
# This discourages the model from memorizing by keeping weights small.
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

# Halve LR if val loss doesn't improve for 2 consecutive epochs
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=2
)

print("Optimizer:", optimizer)
print("Loss:     ", criterion)


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for batch_x, batch_y in tqdm(loader, leave=False, desc="  Training"):
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()
        logits = model(batch_x)                  # (B, num_classes)
        loss   = criterion(logits, batch_y)
        loss.backward()

        # Gradient clipping — prevents exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()

        total_loss += loss.item() * batch_x.size(0)
        preds      = logits.argmax(dim=-1)
        correct    += (preds == batch_y).sum().item()
        total      += batch_x.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0

    for batch_x, batch_y in loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        logits = model(batch_x)
        loss   = criterion(logits, batch_y)

        total_loss += loss.item() * batch_x.size(0)
        preds      = logits.argmax(dim=-1)
        correct    += (preds == batch_y).sum().item()
        total      += batch_x.size(0)

    return total_loss / total, correct / total


print("Training functions defined.")

In [ ]:
# ─── TRAINING LOOP WITH EARLY STOPPING ──────────────────────────────────────

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}
best_val_loss    = float("inf")
best_model_path  = "best_model.pt"
epochs_no_improve = 0  # early stopping counter

print(f"Starting training for up to {EPOCHS} epoch(s) (patience={PATIENCE})...")
print(f"Device: {DEVICE}")
print("-" * 60)

for epoch in range(1, EPOCHS + 1):
    print(f"Epoch {epoch}/{EPOCHS}")

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE)
    val_loss,   val_acc   = evaluate(model, val_loader, criterion, DEVICE)

    scheduler.step(val_loss)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)

    gap = train_acc - val_acc
    print(f"  Train  →  loss: {train_loss:.4f}  |  acc: {train_acc:.4f}")
    print(f"  Val    →  loss: {val_loss:.4f}  |  acc: {val_acc:.4f}")
    print(f"  Gap    →  train_acc - val_acc: {gap:.4f}", "⚠ overfitting" if gap > 0.08 else "✓ healthy")

    # Save best checkpoint
    if val_loss < best_val_loss:
        best_val_loss     = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), best_model_path)
        print(f"  ✓ Best model saved (val_loss={val_loss:.4f})")
    else:
        epochs_no_improve += 1
        print(f"  No improvement ({epochs_no_improve}/{PATIENCE})")
        if epochs_no_improve >= PATIENCE:
            print(f"\nEarly stopping triggered at epoch {epoch}.")
            print(f"Best val_loss: {best_val_loss:.4f}")
            break

    print()

print(f"\nTraining complete. Best val_loss: {best_val_loss:.4f}")
print("Loading best checkpoint for evaluation...")
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
print("Best model loaded ✓")


In [ ]:
# Plot training curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

epochs_x = range(1, len(history["train_loss"]) + 1)

ax1.plot(epochs_x, history["train_loss"], label="Train Loss", marker='o')
ax1.plot(epochs_x, history["val_loss"],   label="Val Loss",   marker='s')
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Loss Curves")
ax1.legend()
ax1.grid(alpha=0.3)

ax2.plot(epochs_x, history["train_acc"], label="Train Acc", marker='o')
ax2.plot(epochs_x, history["val_acc"],   label="Val Acc",   marker='s')
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Accuracy")
ax2.set_title("Accuracy Curves")
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

---
## 15. Evaluation

Load the best checkpoint and evaluate on the **test set** (held out during training).

Metrics:
- **Accuracy** — overall fraction correct
- **Precision** — of predicted positives, how many are truly positive?
- **Recall** — of actual positives, how many did we catch?
- **F1** — harmonic mean of precision and recall
- **Confusion Matrix** — visualize error types

In [ ]:
# Load best model
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.eval()
print("Best model loaded.")

# Collect all predictions
all_preds, all_labels = [], []

with torch.no_grad():
    for batch_x, batch_y in tqdm(test_loader, desc="Evaluating"):
        batch_x = batch_x.to(DEVICE)
        logits  = model(batch_x)
        preds   = logits.argmax(dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(batch_y.numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

# Compute metrics
acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, zero_division=0)
rec  = recall_score(all_labels, all_preds, zero_division=0)
f1   = f1_score(all_labels, all_preds, zero_division=0)
cm   = confusion_matrix(all_labels, all_preds)

print("\n" + "=" * 40)
print("TEST SET RESULTS")
print("=" * 40)
print(f"  Accuracy:  {acc:.4f}  ({acc*100:.2f}%)")
print(f"  Precision: {prec:.4f}")
print(f"  Recall:    {rec:.4f}")
print(f"  F1 Score:  {f1:.4f}")

In [ ]:
# Confusion Matrix
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=['Negative', 'Positive'],
    yticklabels=['Negative', 'Positive'],
    ax=ax
)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True Negatives:  {tn}")
print(f"False Positives: {fp}")
print(f"False Negatives: {fn}")
print(f"True Positives:  {tp}")

---
## 16. Inference

A clean `predict()` function you can use for any new review.

In [ ]:
LABELS = {0: "Negative", 1: "Positive"}

def predict(review: str, model, vocab, max_len, device):
    """
    Predict the sentiment of a single review.

    Returns:
        label:      "Positive" or "Negative"
        confidence: probability of the predicted class (0–100%)
    """
    model.eval()

    tokens   = preprocess(review)
    ids      = vocab.encode(tokens, max_len)
    tensor   = torch.tensor([ids], dtype=torch.long).to(device)   # (1, seq_len)

    with torch.no_grad():
        logits = model(tensor)                      # (1, 2)
        probs  = F.softmax(logits, dim=-1)[0]       # (2,)

    pred_class  = probs.argmax().item()
    confidence  = probs[pred_class].item() * 100

    return LABELS[pred_class], confidence


# ─── TEST IT ─────────────────────────────────────────────────────────────────

test_reviews = [
    "This movie was absolutely fantastic. The acting was superb and the plot kept me engaged throughout.",
    "Terrible film. Boring, predictable, and a complete waste of time. I want my money back.",
    "It was okay, nothing special. Some parts were good but overall pretty average.",
    "One of the best performances I've ever seen. A masterpiece of modern cinema.",
    "Dull, lifeless, and painfully slow. The dialogue was cringeworthy.",
]

print("INFERENCE RESULTS")
print("=" * 70)
for review in test_reviews:
    label, conf = predict(review, model, vocab, MAX_LEN, DEVICE)
    print(f"Review:     {review[:65]}..." if len(review) > 65 else f"Review:     {review}")
    print(f"Prediction: {label} ({conf:.1f}%)")
    print("-" * 70)

In [ ]:
# Interactive cell — try your own review
my_review = "The cinematography was stunning but the story was disappointing."

label, confidence = predict(my_review, model, vocab, MAX_LEN, DEVICE)
print(f"Review:     {my_review}")
print(f"Prediction: {label}")
print(f"Confidence: {confidence:.1f}%")

---
## Bonus: Look Inside the Model

Check what the model has learned by examining attention patterns.

In [ ]:
def get_attention_weights(review: str, model, vocab, max_len, device, block_idx=0):
    """
    Extract attention weights from a specific Transformer block.
    Returns: tokens, attention_weights (num_heads, seq_len, seq_len)
    """
    tokens = preprocess(review)[:max_len]
    ids    = vocab.encode(tokens, max_len)
    tensor = torch.tensor([ids], dtype=torch.long).to(device)

    model.eval()
    attention_maps = []

    # Hook into the attention module
    def hook_fn(module, input, output):
        # Recompute attention weights (module is MultiHeadAttention)
        x = input[0]
        B, S, _ = x.shape
        Q = module.split_heads(module.W_q(x))
        K = module.split_heads(module.W_k(x))
        V = module.split_heads(module.W_v(x))
        _, attn = scaled_dot_product_attention(Q, K, V)
        attention_maps.append(attn.detach().cpu())

    handle = model.encoder.blocks[block_idx].attention.register_forward_hook(hook_fn)

    with torch.no_grad():
        model(tensor)

    handle.remove()
    return tokens, attention_maps[0][0]  # (num_heads, S, S)


# Visualize attention for a sample review
sample_review = "The acting was brilliant but the story was confusing and slow."
tokens, attn_weights = get_attention_weights(
    sample_review, model, vocab, MAX_LEN, DEVICE, block_idx=0
)

# Show first N tokens for readability
N = min(15, len(tokens))
display_tokens = tokens[:N]
display_attn   = attn_weights[:, :N, :N].numpy()

fig, axes = plt.subplots(1, NUM_HEADS, figsize=(4 * NUM_HEADS, 4))
if NUM_HEADS == 1:
    axes = [axes]

for h, ax in enumerate(axes):
    im = ax.imshow(display_attn[h], cmap='viridis', vmin=0)
    ax.set_xticks(range(N))
    ax.set_yticks(range(N))
    ax.set_xticklabels(display_tokens, rotation=90, fontsize=7)
    ax.set_yticklabels(display_tokens, fontsize=7)
    ax.set_title(f"Head {h+1}")
    plt.colorbar(im, ax=ax)

plt.suptitle(f"Attention Weights (Block 1)\n\"{sample_review[:60]}...\"")
plt.tight_layout()
plt.show()

---
## 17. Save & Download Model

Three files are needed to rebuild the model in your app:
- `sentiment_transformer.pt` — trained weights
- `vocab.pkl` — the vocabulary (word → index mapping)
- `model_config.json` — hyperparameters to reconstruct the architecture

**Option A:** Download directly to your computer (browser download)  
**Option B:** Save to Google Drive (persists across Colab sessions)

In [ ]:
# ─── SAVE MODEL, VOCAB & CONFIG ──────────────────────────────────────────────
import pickle, json as _json

# 1. Model weights
torch.save(model.state_dict(), "sentiment_transformer.pt")

# 2. Vocabulary
with open("vocab.pkl", "wb") as f:
    pickle.dump(vocab, f)

# 3. Config — everything needed to reconstruct the model architecture
config = {
    "vocab_size":  vocab.size,
    "embed_dim":   EMBED_DIM,
    "num_heads":   NUM_HEADS,
    "ff_dim":      FF_DIM,
    "num_blocks":  NUM_BLOCKS,
    "max_len":     MAX_LEN,
    "num_classes": NUM_CLASSES,
    "dropout":     DROPOUT,
}
with open("model_config.json", "w") as f:
    _json.dump(config, f, indent=2)

print("Saved:")
print("  sentiment_transformer.pt  — model weights")
print("  vocab.pkl                 — vocabulary")
print("  model_config.json         — architecture config")


In [ ]:
# ─── DOWNLOAD FILES (Google Colab) ───────────────────────────────────────────
# Run this cell to download all three files to your computer.
# Skip if you're not on Colab.

try:
    from google.colab import files
    print("Downloading model files...")
    files.download("sentiment_transformer.pt")
    files.download("vocab.pkl")
    files.download("model_config.json")
    print("Done — check your browser downloads.")
except ImportError:
    print("Not running on Colab — files are saved in the current directory.")


In [ ]:
# ─── SAVE TO GOOGLE DRIVE (optional, survives session restarts) ───────────────
# Mounts your Drive and copies all three files into MyDrive/review-transformer/
# Useful so you don't have to retrain if the Colab session disconnects.

import shutil, os

try:
    from google.colab import drive
    drive.mount("/content/drive")

    save_dir = "/content/drive/MyDrive/review-transformer"
    os.makedirs(save_dir, exist_ok=True)

    for fname in ["sentiment_transformer.pt", "vocab.pkl", "model_config.json"]:
        shutil.copy(fname, os.path.join(save_dir, fname))
        print(f"  Copied {fname} → {save_dir}/{fname}")

    print("\nAll files saved to Google Drive.")
except ImportError:
    print("Not running on Colab — skipping Drive mount.")


In [ ]:
# ─── HOW TO RELOAD IN YOUR APP ───────────────────────────────────────────────
# Copy this snippet into your FastAPI / Streamlit app.
# You'll also need to copy the model class definitions (or import from models/).

RELOAD_SNIPPET = '''
import json, pickle, torch

# 1. Load config
with open("model_config.json") as f:
    config = json.load(f)

# 2. Load vocabulary
with open("vocab.pkl", "rb") as f:
    vocab = pickle.load(f)

# 3. Rebuild model and load weights
model = SentimentClassifier(**config)
model.load_state_dict(torch.load("sentiment_transformer.pt", map_location="cpu"))
model.eval()

# 4. Predict
label, confidence = predict("The movie was fantastic!", model, vocab, config["max_len"], "cpu")
print(label, confidence)
'''

print(RELOAD_SNIPPET)


---
## What You've Built

| Component | What it does |
|---|---|
| `Vocabulary` | Maps words ↔ integer ids, handles OOV with `<UNK>` |
| `TokenEmbedding` | Lookup table: token id → dense vector |
| `PositionalEncoding` | Adds sinusoidal position signal to embeddings |
| `scaled_dot_product_attention` | Core attention: Q·K → weights → weighted V |
| `MultiHeadAttention` | H parallel attention heads, merged via output projection |
| `FeedForward` | Position-wise MLP with ReLU |
| `TransformerBlock` | Attention + FFN with residual connections & LayerNorm |
| `TransformerEncoder` | Stacked blocks with padding mask |
| `SentimentClassifier` | Encoder + mean pooling + linear head |
